In [ ]:
import h5py
import numpy as np
h5_path = "NSC1_Simple Segmentation.h5"

def print_h5_structure(name, obj):
    print(name, type(obj), getattr(obj, 'shape', ''))

with h5py.File(h5_path, "r") as f:
    f.visititems(print_h5_structure)


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

folder = Path("./ilastik")
output_csv = folder / "gfp_leaf_ratios.csv"
dataset_key = "exported_data"   # change if needed

results = []

for h5_path in sorted(folder.glob("*.h5")):
    try:
        with h5py.File(h5_path, "r") as f:
            seg = f[dataset_key][...]

        # remove singleton dimension if needed
        if seg.ndim == 3 and seg.shape[0] == 1:
            seg = seg[0]

        unique_vals = np.unique(seg)

        # Example mapping:
        # 1 = background
        # 2 = leaf
        # 3 = GFP
        leaf_mask = (seg == 2)
        gfp_mask = (seg == 3)
        total_leaf_mask = leaf_mask | gfp_mask

        leaf_area = int(total_leaf_mask.sum())
        gfp_area = int(gfp_mask.sum())
        ratio = gfp_area / leaf_area if leaf_area > 0 else np.nan

        results.append({
            "file": h5_path.name,
            "unique_labels": ",".join(map(str, unique_vals)),
            "leaf_area_px": leaf_area,
            "gfp_area_px": gfp_area,
            "gfp_leaf_ratio": ratio,
            "gfp_leaf_percent": ratio * 100 if not np.isnan(ratio) else np.nan,
        })

    except Exception as e:
        results.append({
            "file": h5_path.name,
            "unique_labels": "",
            "leaf_area_px": np.nan,
            "gfp_area_px": np.nan,
            "gfp_leaf_ratio": np.nan,
            "gfp_leaf_percent": np.nan,
            "error": str(e)
        })

df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)

print(df)
print(f"Saved to: {output_csv}")